## 1. Imports 


In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import onnxruntime as ort

# Add project root to path for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..")) if "__file__" in dir() else os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from config import LIVENESS_MODEL_PATH, FACE_MODEL_DIR
from model.onnx_providers import get_providers

# Output directory for this notebook
OUTPUT_DIR = os.path.join(os.getcwd(), "outputs", "liveness")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete")
print(f"Model: {LIVENESS_MODEL_PATH}")
print(f"Output: {OUTPUT_DIR}")

Setup complete
Model: C:\Users\calvi\Downloads\Project\BE_Project\BE_Project_V2\Other Models\liveness\MiniFASNetV2.onnx
Output: C:\Users\calvi\Downloads\Project\BE_Project\BE_Project_V2\working on models\outputs\liveness


## 2. Load Model
Load the ONNX anti-spoofing model.

In [3]:
providers = get_providers()
print(f"Using providers: {providers}")

session = ort.InferenceSession(LIVENESS_MODEL_PATH, providers=providers)
input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape

print(f"Model loaded: {input_name}")
print(f"Input shape: {input_shape}")
print(f"Classes: ['print', 'live', 'replay']")

Using providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Model loaded: input
Input shape: ['batch_size', 3, 80, 80]
Classes: ['print', 'live', 'replay']


## 3. Helper Functions
Functions for face detection, crop transformation, and prediction.

In [4]:
def transform_crop(img_bgr, bbox, scale=2.7, target_size=80):
    """Crop and transform face region for liveness detection."""
    h, w = img_bgr.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    side = max(x2 - x1, y2 - y1)
    crop_side = side * scale

    cx0 = int(cx - crop_side / 2)
    cy0 = int(cy - crop_side / 2)
    x1c, y1c = max(cx0, 0), max(cy0, 0)
    x2c, y2c = min(cx0 + int(crop_side), w), min(cy0 + int(crop_side), h)

    crop = img_bgr[y1c:y2c, x1c:x2c]

    pad_top = max(cy0 - 0, 0)
    pad_bottom = max(int(crop_side) - (y2c - y1c) - pad_top, 0)
    pad_left = max(cx0 - 0, 0)
    pad_right = max(int(crop_side) - (x2c - x1c) - pad_left, 0)
    crop = cv2.copyMakeBorder(
        crop, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REPLICATE
    )

    crop = cv2.resize(crop, (target_size, target_size), interpolation=cv2.INTER_AREA)
    crop = crop.astype(np.float32)
    crop = crop.transpose(2, 0, 1)[None]  # 1,3,80,80
    return crop


def softmax(x):
    """Compute softmax probabilities."""
    x = np.asarray(x, dtype=np.float32)
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


def predict_liveness(img_bgr, bbox):
    """Predict liveness for a face crop."""
    inp = transform_crop(img_bgr, bbox)
    out = session.run(None, {input_name: inp})[0]
    probs = softmax(out[0])
    class_idx = int(np.argmax(probs))
    classes = ["print", "live", "replay"]
    return classes[class_idx], probs[class_idx], probs


def check_blur(img_bgr, bbox, threshold=80):
    """Check if face region is too blurry (Laplacian variance)."""
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1c, y1c = max(0, x1), max(0, y1)
    x2c, y2c = min(img_bgr.shape[1], x2), min(img_bgr.shape[0], y2)
    face_crop = img_bgr[y1c:y2c, x1c:x2c]
    if face_crop.size == 0:
        return False, 0
    gray = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    return laplacian_var < threshold, laplacian_var


print("Helpers ready")

Helpers ready


## 4. Load Test Images
Load test samples for liveness evaluation.

In [ ]:
from insightface.app import FaceAnalysis
from config import INSIGHTFACE_NAME, INSIGHTFACE_ROOT

face_app = FaceAnalysis(name=INSIGHTFACE_NAME, root=INSIGHTFACE_ROOT,
                        providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
face_app.prepare(ctx_id=-1, det_size=(640, 640))

test_dir = os.path.join(PROJECT_ROOT, "Test_Samples")
test_images = {}
for fname in os.listdir(test_dir):
    if fname.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(test_dir, fname)
        img = cv2.imread(path)
        if img is not None:
            test_images[fname] = img

print(f"Loaded {len(test_images)} test images")
for name in test_images:
    print(f"  {name}: {test_images[name].shape}")

## 5. Run Liveness Detection
Test the model on real images.

In [ ]:
results = []

for name, img in test_images.items():
    padded = cv2.copyMakeBorder(img, 50, 50, 50, 50, cv2.BORDER_REPLICATE)
    faces = face_app.get(padded)
    
    if not faces:
        print(f"{name}: No face detected")
        continue
    
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    bbox = [int(v) for v in face.bbox]
    
    pred_class, pred_conf, probs = predict_liveness(img, bbox)
    is_blur, blur_val = check_blur(img, bbox)
    
    results.append({
        "name": name,
        "prediction": pred_class,
        "confidence": pred_conf,
        "probs": probs,
        "blurry": is_blur,
        "blur_value": blur_val
    })
    
    print(f"{name}: {pred_class} ({pred_conf:.2%}), blur={blur_val:.1f}")

print(f"\nTotal: {len(results)} images processed")

## 6. Visualize Results
Show predictions with probability bars.

In [ ]:
classes = ["print", "live", "replay"]
colors_map = {"print": "red", "live": "green", "replay": "orange"}

fig, axes = plt.subplots(1, min(4, len(results)), figsize=(4 * min(4, len(results)), 4))
if len(results) == 1:
    axes = [axes]

for idx, res in enumerate(results[:4]):
    ax = axes[idx]
    probs = res["probs"]
    bars = ax.barh(classes, probs, color=[colors_map[c] for c in classes])
    ax.set_xlim(0, 1)
    ax.set_title(f"{res['name']}\nPred: {res['prediction']} ({res['confidence']:.1%})")
    ax.bar_label(bars, fmt="%.2f")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "liveness_predictions.png"), dpi=150)
plt.show()

## 7. Liveness Pipeline Summary
The liveness detector uses 3-layer defense:
1. **CNN Model** - MiniFASNetV2 classifies face as live/print/replay
2. **Laplacian Blur Check** - Rejects blurry images (variance < 80)
3. **Temporal Smoothing** - Requires 2/3 frames to agree on 'live'

In [ ]:
print("Liveness Detection Pipeline:")
print("=" * 50)
print("1. CNN Model: MiniFASNetV2 (ONNX)")
print("   - Input: 1x3x80x80 BGR crop")
print("   - Output: [print, live, replay] probabilities")
print("   - Threshold: >= 0.5 confidence for 'live'")
print()
print("2. Laplacian Blur Check")
print("   - Computes variance of Laplacian on face grayscale")
print("   - Rejects if variance < 80 (too blurry)")
print()
print("3. Temporal Smoothing")
print("   - Keeps history of last 3 predictions")
print("   - Requires >= 2 out of 3 frames to be 'live'")
print()
print("Attack Types:")
print("  - print: Printed photo held in front of camera")
print("  - replay: Screen replay (phone/laptop display)")
print("  - live: Real person")